# 1. Reading PDFs

In [2]:
import os
file_path = "../../data/examples/week_7/imf_reports"
os.listdir(file_path)

['Angola_2026.pdf', 'Antigua_2026.pdf']

## 1.1 Read: Pypdf2

In [3]:
import os
from PyPDF2 import PdfReader

def read_pdf(file_path):
    result = {}
    with open(file_path, 'rb') as f:
        pdf = PdfReader(f)
        for page_num, page in enumerate(pdf.pages, start=1):
            result[f"Page {page_num}"] = page.extract_text()
    return result

# Usage
imf_antigua = read_pdf(f"{file_path}/Antigua_2026.pdf")
imf_antigua["Page 1"][:100]

' \n© 2026 International Monetary Fund  \nIMF Country Report No.  26/97 \nANTIGUA AND BARBUDA  \n2026 ART'

## 1.2 Read: pdfplumber

In [4]:
import pdfplumber
import os

def read_pdf(file_path):
    result = {}
    
    with pdfplumber.open(file_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            result[f"Page {page_num}"] = text if text else ""
    
    return result

# Usage
imf_antigua = read_pdf(f"{file_path}/Antigua_2026.pdf")
imf_antigua["Page 1"][:100]

'IMF Country Report No. 26/97\nANTIGUA AND BARBUDA\n2026 ARTICLE IV CONSULTATION; PRESS RELEASE;\nMay 20'

# 3. Generative AI

## 3.1 On your local machine

Ollama runs open-source models locally. Start Ollama on your computer first, then choose a model you have already pulled. This example uses `llama3.2:latest`, which is installed on this machine.

In [5]:
import requests

OLLAMA_MODEL = "llama3.2:latest"

def llm_ollama(input_text, model=OLLAMA_MODEL):
    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [{"role": "user", "content": input_text}],
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["message"]["content"]

In [8]:
ollama_response = llm_ollama("Tell me a fun fact about Japan.")
print(ollama_response)

Here's a fun fact about Japan:

In Japan, there is a vending machine that dispenses live baby chicks! Yes, you read that right! These "chick vending machines" are a popular novelty in Japan, and they can be found in many cities across the country. The machines are typically filled with fertilized eggs from local farms, which hatch into adorable baby chicks about 10-14 days later. When the egg hatches, the chick is carefully extracted and handed to the customer, who then gets to take care of their new feathered friend until it's old enough to be released back into the wild. It's a unique and entertaining experience that's definitely worth trying if you ever visit Japan!


## 3.2 Using closed source model: Gemini

- https://ai.google.dev/gemini-api/docs

In [9]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
gemini_api_key = os.getenv("API_GEMINI")

if not gemini_api_key:
    raise ValueError("Add API_GEMINI to your .env file before running this cell.")

In [10]:
def llm_gemini(input_text, gemini_model="gemini-3.5-flash"):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{gemini_model}:generateContent"
    response = requests.post(
        url,
        params={"key": gemini_api_key},
        json={"contents": [{"parts": [{"text": input_text}]}]},
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

In [11]:
gemini_response = llm_gemini("Tell me a fun fact about Japan.")
print(gemini_response)

Here is a fun fact about Japan:

**Japan has "blue" traffic lights instead of green!** 

In Japanese, the word for blue (**ao**) was historically used to describe both blue and green objects (for example, green apples and green grass were described as *ao*). 

When traffic lights were first introduced in Japan in the 1930s, the "go" light was green. However, the public kept referring to it as *ao* (blue) rather than *midori* (the modern Japanese word for green). 

To compromise, the Japanese government made a unique law in 1973: traffic lights had to use the absolute bluest shade of green possible. Because of this, when you visit Japan, you will notice that the "go" light is a distinct, beautiful shade of turquoise-blue!


## 3.3 Using closed source model: Claude

- https://docs.anthropic.com/en/docs/about-claude/models/overview

In [12]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
claude_api_key = os.getenv("API_CLAUDE")

if not claude_api_key:
    raise ValueError("Add API_CLAUDE to your .env file before running this cell.")

client = anthropic.Anthropic(api_key=claude_api_key)

In [13]:

def llm_claude(input_text, claude_model="claude-haiku-4-5"):
    messages = [
        {"role": "user", "content": input_text}
    ]
    chat_completion = client.messages.create(
        model=claude_model,
        max_tokens=100,
        messages=messages
    )
    return chat_completion

# Fixed function call - using the correct function name
claude_response = llm_claude(
    input_text="Tell me a fun fact about Japan.", 
    claude_model="claude-sonnet-4-6"
)

# Access the response content
print(claude_response.content[0].text)

Here's a fun fact about Japan:

**Japan has more vending machines per capita than any other country in the world!** There are approximately **5 million vending machines** across Japan — roughly one for every 23 people. And they sell much more than just drinks — you can find machines selling hot ramen, fresh eggs, umbrellas, flowers, and even neckties! 🍜🌸
